# OpenAI Chat provider: the agent, end to end

The same agent the Anthropic notebook drives (`AnthropicAgent`), running on a model in the **OpenAI Chat Completions** format through **Vercel AI Gateway**: tools, a resumed session, an answer check, structured output, reasoning, a frontend tool's pause, sub-agents, compaction and externalization, abort and steer, and the bill. Only the provider changes; the turn loop is the one Claude runs on.

```python
router = ModelRouter(others=VercelGateway.from_env())                  # Claude would go direct
agent = AnthropicAgent(model=MODEL, provider_value=router.provider_for(MODEL), tools=[...])
```

**Live, and free by default.** Every section calls the real gateway with `AI_GATEWAY_API_KEY` from `.env`, on **Nemotron 3.5 Lightning Free** (sub-agents also use **Flint v1.0**): both are free per token, so the notebook costs $0. They keep no zero data retention, so the prompts are made up. The image section needs a model that reads images: set `RUN_CREDIT = True` to run it on Gemini 2.5 Flash, which the free tier's $5 a month covers (a fraction of a cent).

Models answer differently from run to run, so each section prints `PASS` or `FAIL` for what it expects instead of stopping, and the last cell sums them up. For the route itself, offline and one part at a time, see `agent_base/providers/vercel_gateway/vercel_gateway_test.ipynb`.

In [1]:
from dotenv import load_dotenv
import asyncio
import json
import os
import tempfile
from pathlib import Path

load_dotenv("../../../.env")

from agent_base.core.messages import Message
from agent_base.core.provider import RetryPolicy
from agent_base.providers.anthropic import AnthropicAgent
from agent_base.providers.openai_chat import OpenAIChatLLMConfig
from agent_base.providers.router import ModelRouter
from agent_base.providers.vercel_gateway import VercelGateway
from agent_base.sandbox.local import LocalSandbox
from agent_base.storage import create_adapters
from agent_base.tools import tool

assert os.environ.get("AI_GATEWAY_API_KEY"), "Set AI_GATEWAY_API_KEY in .env (see .env.example)."

MODEL = os.environ.get("OPENAI_CHAT_NOTEBOOK_MODEL", "nvidia/nemotron-3.5-lightning-free")
SECOND_MODEL = "recursive/flint-v1.0"          # free too: the sub-agents' model
IMAGE_MODEL = "google/gemini-2.5-flash"         # reads images; on the free tier's monthly credit
RUN_CREDIT = False                              # True runs the image section on IMAGE_MODEL
RETRY = RetryPolicy(max_retries=4, base_delay=2.0)  # the free tier's rate limits are per model

gateway = VercelGateway.from_env()
router = ModelRouter(others=gateway)
WORK = Path(tempfile.gettempdir()) / "openai_chat_agent_notebook"
WORK.mkdir(exist_ok=True)


def provider(model=MODEL):
    return router.provider_for(model, retry_policy=RETRY)


def adapters(name):
    return create_adapters("filesystem", base_path=str(WORK / name))


def sandbox_in_work(agent_uuid):
    return LocalSandbox(sandbox_id=agent_uuid, base_dir=WORK / "sandboxes")


CHECKS = []

def check(ok, what):
    CHECKS.append((bool(ok), what))
    print(("PASS " if ok else "FAIL ") + what)


print("model:", MODEL, "| record:", provider().catalog.get(MODEL))

model: nvidia/nemotron-3.5-lightning-free | record: ModelRecord(id='nvidia/nemotron-3.5-lightning-free', api='chat_completions', context_tokens=1000000, output_tokens=65000, inputs=frozenset({'text'}), reasoning=True, zero_data_retention=False, tools=True)


In [2]:
# ── Test Utils ─────────────────────────────────────────────────────
#
# As in the Anthropic agent notebook: the loop emits typed StreamItems (content deltas and MetaEnvelope control
# frames) into the agent's stream queue. A fresh queue is attached per run, and agent.run() drives the turn.

_DONE = object()


def _render_item(item) -> str:
    kind = getattr(item, "kind", None)
    if kind is not None:
        return f"[meta:{kind}] {str(item.body)[:160]}"
    t = getattr(item, "type", "")
    if t == "text":
        return item.text
    if t == "thinking":
        return f"[thinking] {item.thinking}"
    if t == "tool_call":
        return f"[tool_call {item.tool_name}] {item.arguments_json}"
    if t == "tool_result":
        return f"[tool_result {item.tool_name}] {str(item.result_content)[:200]}"
    return repr(item)


def _attach_fresh_queue(agent) -> asyncio.Queue:
    queue: asyncio.Queue = asyncio.Queue()
    agent._stream_queue = queue
    return queue


def _start_runner(agent, prompt, queue):
    async def _runner():
        try:
            return await agent.run(prompt)
        finally:
            queue.put_nowait(_DONE)
    return asyncio.create_task(_runner())


async def run_with_stream(agent, prompt, *, print_items=True):
    """Drive one full turn while printing the typed stream; returns the AgentResult."""
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)
    items = []
    while True:
        item = await queue.get()
        if item is _DONE:
            break
        items.append(item)
        if print_items:
            print(_render_item(item), end="" if getattr(item, "type", "") == "text" else "\n", flush=True)
    print()
    result = await runner
    result.stream_items = items
    return result


async def run_until_pause_or_done(agent, prompt):
    """Start a run and drain until it completes or parks on a frontend tool's pause (an await_input frame)."""
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)
    while True:
        item = await queue.get()
        if item is _DONE:
            return runner, None
        print(_render_item(item), flush=True)
        if getattr(item, "kind", None) == "await_input":
            return runner, item


async def finish_run(agent, runner):
    queue = agent._stream_queue
    while True:
        item = await queue.get()
        if item is _DONE:
            break
        print(_render_item(item), flush=True)
    return await runner


class DemoHandle:
    """An in-flight streaming run, to abort or steer."""

    def __init__(self, agent, runner, collector):
        self.agent, self._runner, self._collector = agent, runner, collector

    async def _settle(self, result):
        if not self._runner.done():
            await self._runner
        if not self._collector.done():
            await self._collector
        print(f"\nstop_reason={result.stop_reason} was_aborted={getattr(result, 'was_aborted', None)} "
              f"abort_phase={getattr(result, 'abort_phase', None)} final_answer={result.final_answer!r:.120}")
        return result

    async def abort(self):
        return await self._settle(await self.agent.abort())

    async def steer(self, instruction):
        return await self._settle(await self.agent.steer(instruction))


async def start_demo(agent, prompt, *, wait_for_frames=3):
    """Start a streaming run, wait for its first frames, and return a handle to abort or steer it."""
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)
    seen = asyncio.Event()
    count = {"n": 0}

    async def _collect():
        while True:
            item = await queue.get()
            if item is _DONE:
                seen.set()
                break
            count["n"] += 1
            if count["n"] >= wait_for_frames:
                seen.set()
            print(_render_item(item), end="" if getattr(item, "type", "") == "text" else "\n", flush=True)

    collector = asyncio.create_task(_collect())
    await asyncio.wait_for(seen.wait(), 120)
    return DemoHandle(agent, runner, collector)

print("ready")

ready


# 1. Tool schemas in the Chat Completions format

`@tool` functions become `function` tools; the docstring is the description and the signature the JSON schema.

In [3]:
from agent_base.providers.openai_chat.formatters import format_tools
from agent_base.tools import ToolRegistry

@tool
def add(a: int, b: int) -> str:
    """Add two integers."""
    return str(a + b)

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two integers."""
    return str(a * b)

registry = ToolRegistry()
registry.register_tools([add, multiply])
wire = format_tools(registry.get_schemas())
print(json.dumps(wire, indent=1))
check([t["function"]["name"] for t in wire] == ["add", "multiply"], "both tools become functions")

[
 {
  "type": "function",
  "function": {
   "name": "add",
   "description": "Add two integers.",
   "parameters": {
    "type": "object",
    "properties": {
     "a": {
      "type": "integer",
      "description": "Parameter of type integer"
     },
     "b": {
      "type": "integer",
      "description": "Parameter of type integer"
     }
    },
    "required": [
     "a",
     "b"
    ]
   }
  }
 },
 {
  "type": "function",
  "function": {
   "name": "multiply",
   "description": "Multiply two integers.",
   "parameters": {
    "type": "object",
    "properties": {
     "a": {
      "type": "integer",
      "description": "Parameter of type integer"
     },
     "b": {
      "type": "integer",
      "description": "Parameter of type integer"
     }
    },
    "required": [
     "a",
     "b"
    ]
   }
  }
 }
]
PASS both tools become functions


# 2. A basic agent with tools

First without a stream (`agent.run`), then with the typed stream, as a host's UI reads it.

In [4]:
basic_store = adapters("basic")
basic_agent = AnthropicAgent(
    system_prompt="Always use the math tools for arithmetic. Reply with the number only.",
    model=MODEL, provider_value=provider(), tools=[add, multiply],
    config=OpenAIChatLLMConfig(max_tokens=4000),
    config_adapter=basic_store[0], conversation_adapter=basic_store[1], run_adapter=basic_store[2],
)
await basic_agent.initialize()
basic = await basic_agent.run("What is 5 + (10 * 2)? Use the tools.")
print("answer:", basic.final_answer, "| steps:", basic.total_steps, "| stop:", basic.stop_reason)
check("25" in basic.final_answer, "the tools give 25")
check(basic.total_steps >= 2, "it took a tool step and an answer step")

print("\n--- with the typed stream ---")
streamed = await run_with_stream(AnthropicAgent(system_prompt="Reply briefly.", model=MODEL, provider_value=provider(),
                                                config=OpenAIChatLLMConfig(max_tokens=2000)),
                                 "Reply with exactly the word PONG.")
check("PONG" in streamed.final_answer.upper(), "the streamed reply is PONG")
check(any(getattr(i, "type", "") == "text" for i in streamed.stream_items), "text frames streamed")

2026-11-18T13:08:19.900073Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:08:20.407445Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:08:21.012183Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


answer: 25 | steps: 3 | stop: end_turn
PASS the tools give 25
PASS it took a tool step and an answer step

--- with the typed stream ---
[meta:run_started] RunStarted(user_query='Reply with exactly the word PONG.', model='nvidia/nemotron-3.5-lightning-free', conversation_log=None)


2026-11-18T13:08:21.247855Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:09:00.766659Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


... (62 outputs left out)

ONG

[meta:usage_report] UsageReport(kind='usage_report', usage={'input_tokens': 27, 'output_tokens': 151, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'thinking_tokens': 147, '_v':


[meta:run_completed] RunCompleted(stop_reason='end_turn', total_steps=1, generated_files=None, cost={'total_cost': 0.0, 'currency': 'USD', 'breakdown': {'reported_cost': 0.0}, 'run_



PASS the streamed reply is PONG
PASS text frames streamed


# 3. Resuming a session from storage

A new agent with the same `agent_uuid` and storage picks the conversation up where it was; the provider is a value, so it is passed again.

In [5]:
resumed_agent = AnthropicAgent(
    agent_uuid=basic_agent.agent_uuid, model=MODEL, provider_value=provider(), tools=[add, multiply],
    config=OpenAIChatLLMConfig(max_tokens=4000),
    config_adapter=basic_store[0], conversation_adapter=basic_store[1], run_adapter=basic_store[2],
)
await resumed_agent.initialize()
resumed = await resumed_agent.run("What result did you give me last time? Reply with the number only.")
print("answer:", resumed.final_answer)
check("25" in resumed.final_answer, "the resumed session remembers 25")

2026-11-18T13:09:02.040456Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


answer: 25
PASS the resumed session remembers 25


# 4. An answer check, and structured output

An `end_turn_hook` checks the final answer and sends the turn back with a correction when it fails, on any provider. `response_format` asks the model for JSON against a schema; on models other than OpenAI's the schema isn't enforced, so the prompt names the fields too.

In [6]:
from agent_base.core.end_turn_hook import EndTurnContext, EndTurnHookResult

def json_with_name_and_description(ctx: EndTurnContext) -> EndTurnHookResult:
    try:
        data = json.loads(ctx.final_text.strip().removeprefix("```json").removesuffix("```").strip())
        if isinstance(data, dict) and {"name", "description"} <= set(data):
            return EndTurnHookResult(action="pass")
    except json.JSONDecodeError:
        pass
    return EndTurnHookResult(action="retry", rollback_message=(
        "Reply with ONLY a JSON object with the fields 'name' and 'description'."))

schema = {"type": "json_schema", "json_schema": {"name": "thing", "schema": {
    "type": "object", "properties": {"name": {"type": "string"}, "description": {"type": "string"}},
    "required": ["name", "description"], "additionalProperties": False}}}
checked_agent = AnthropicAgent(
    system_prompt='Return JSON only: {"name": ..., "description": ...}.', model=MODEL, provider_value=provider(),
    config=OpenAIChatLLMConfig(max_tokens=2000, response_format=schema), end_turn_hook=json_with_name_and_description,
)
checked = await checked_agent.run("Describe a calculator.")
print("answer:", checked.final_answer)
parsed = json.loads(checked.final_answer.strip().removeprefix("```json").removesuffix("```").strip())
check({"name", "description"} <= set(parsed), "the answer is JSON with name and description")

2026-11-18T13:09:03.911421Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


answer: {"name": "Calculator", "description": "An electronic or mechanical device used to perform arithmetic operations such as addition, subtraction, multiplication, and division, often featuring advanced functions for science, engineering, and finance."}
PASS the answer is JSON with name and description


# 5. Classification: one call, or a forced tool

Classification needs no agent loop: one `generate` call with `response_format`. Or the label can be read from a tool call that `tool_choice` forces on the first step (once a result comes back the model chooses, so an agent can still finish).

In [7]:
label_schema = {"type": "json_schema", "json_schema": {"name": "sentiment", "strict": True, "schema": {
    "type": "object", "properties": {"label": {"type": "string", "enum": ["positive", "negative", "neutral"]}},
    "required": ["label"], "additionalProperties": False}}}
turn = await provider().generate(
    system_prompt='Classify the sentiment of the review. Reply with JSON only: {"label": "positive" | "negative" | "neutral"}.',
    messages=[Message.user("The soup was cold and the waiter ignored us for an hour.")], tool_schemas=[],
    llm_config=OpenAIChatLLMConfig(max_tokens=2000, response_format=label_schema), model=MODEL)
label = turn.message.content[-1].text
print("one call:", label)
check(json.loads(label) == {"label": "negative"}, "one call labels the review negative")

@tool
def record_label(label: str) -> str:
    """Record the review's sentiment: positive, negative or neutral."""
    return f"recorded {label}"

forced_agent = AnthropicAgent(system_prompt="Classify each review with record_label, then say done.", model=MODEL,
                              provider_value=provider(), tools=[record_label],
                              config=OpenAIChatLLMConfig(max_tokens=2000, tool_choice="record_label"))
forced = await forced_agent.run("Review: The view was lovely and the staff were kind.")
calls = [b for m in forced_agent.agent_config.context_messages for b in m.content if type(b).__name__ == "ToolUseContent"]
print("forced tool:", [(c.tool_name, c.tool_input) for c in calls], "| then:", forced.final_answer[:80])
check(calls and calls[0].tool_input.get("label") == "positive", "the forced call labels the review positive")

2026-11-18T13:09:04.110322Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:10:01.839887Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


one call: {"label": "negative"}
PASS one call labels the review negative


2026-11-18T13:10:02.453317Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:10:03.094326Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


forced tool: [('record_label', {'label': 'positive'})] | then: done
PASS the forced call labels the review positive


# 6. Reasoning

`effort` becomes the request's `reasoning.effort`. The model's reasoning streams as `thinking` frames, and its `reasoning_details` go back on the next run, to this model only.

In [8]:
thinker = AnthropicAgent(system_prompt="Answer in one sentence.", model=MODEL, provider_value=provider(), tools=[multiply],
                         config=OpenAIChatLLMConfig(max_tokens=4000, effort="low"))
first = await run_with_stream(thinker, "What is 37 * 25? Use the tool.", print_items=False)
thought = [i for i in first.stream_items if getattr(i, "type", "") == "thinking" and i.thinking]
print("reasoning frames:", len(thought), "| e.g.", (thought[0].thinking[:100] if thought else ""))
second = await thinker.run("And what is that plus 75? Use the add tool if you have one, else compute it.")
kept = [b for m in thinker.agent_config.context_messages for b in m.content
        if type(b).__name__ == "ThinkingContent" and b.kwargs.get("reasoning_details")]
print("answers:", first.final_answer, "|", second.final_answer)
check(thought, "reasoning streamed as thinking frames")
check(kept, "the reasoning was kept to send back on the next run")
digits = lambda text: "".join(ch for ch in text if ch.isdigit())  # "1,000" and "1 000" both read as 1000
check("925" in digits(first.final_answer) and "1000" in digits(second.final_answer), "37 * 25 = 925, plus 75 = 1000")

2026-11-18T13:10:03.311139Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:11:01.253275Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:11:11.752005Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]



reasoning frames: 96 | e.g. Here


2026-11-18T13:11:12.492983Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


answers: 37 multiplied by 25 equals 925. | 925 plus 75 equals 1000.
PASS reasoning streamed as thinking frames
PASS the reasoning was kept to send back on the next run
PASS 37 * 25 = 925, plus 75 = 1000


# 7. An image in the message (spends credit)

Images go as `image_url` parts with a data URL. The free models read text only, so this runs on `IMAGE_MODEL` when `RUN_CREDIT = True`.

In [9]:
import base64, struct, zlib

def png(width, height, rgb):
    raw = b"".join(b"\x00" + bytes(rgb) * width for _ in range(height))
    def chunk(kind, data):
        return struct.pack(">I", len(data)) + kind + data + struct.pack(">I", zlib.crc32(kind + data) & 0xffffffff)
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b""))

if not RUN_CREDIT:
    print("Skipped: set RUN_CREDIT = True to run on", IMAGE_MODEL, "(the free tier's monthly credit).")
else:
    from agent_base.core.types import ImageContent, TextContent
    red = base64.b64encode(png(16, 16, (220, 30, 30))).decode()
    seer = AnthropicAgent(system_prompt="Answer in one word.", model=IMAGE_MODEL, provider_value=provider(IMAGE_MODEL),
                          config=OpenAIChatLLMConfig(max_tokens=1000))
    seen = await seer.run(Message.user([ImageContent(source_type="base64", media_type="image/png", data=red),
                                        TextContent(text="What colour is this image?")]))
    print("answer:", seen.final_answer, "| cost: $", seen.settlement.turn_cost.total_cost)
    check("red" in seen.final_answer.lower(), "the model sees a red image")

Skipped: set RUN_CREDIT = True to run on google/gemini-2.5-flash (the free tier's monthly credit).


# 8. A frontend tool: the run pauses, the reply resumes it

A tool with `executor="frontend"` runs in the client. The run parks on an `await_input` frame with a correlation id (`cid`); the client's `ToolReply(cid, results)` wakes it, and the model gets the result.

In [10]:
from agent_base.core.commands import ToolReply
from agent_base.core.types import ToolResultContent

@tool(executor="frontend")
def user_confirm(message: str) -> str:
    """Ask the user for yes/no confirmation before going on."""
    pass

relay_store = adapters("relay")
relay_agent = AnthropicAgent(
    system_prompt="For every request, first call user_confirm with a short question. Answer only after it returns.",
    model=MODEL, provider_value=provider(), frontend_tools=[user_confirm], config=OpenAIChatLLMConfig(max_tokens=2000),
    config_adapter=relay_store[0], conversation_adapter=relay_store[1], run_adapter=relay_store[2],
)
runner, pause = await run_until_pause_or_done(relay_agent, "Say hello to me.")
check(pause is not None, "the run paused for user_confirm")
if pause is not None:
    call = pause.body.tools[0]
    print(f"\npaused on cid {pause.correlation_id}: {call.tool_name}({call.input})")
    check(relay_agent.agent_config.pending_relay.cid == pause.correlation_id, "the pause is saved for a cold resume")
    ack = await relay_agent.submit(ToolReply(cid=pause.correlation_id, results=[
        ToolResultContent(tool_id=call.tool_use_id, tool_name=call.tool_name, tool_result="yes")]))
    print("ack:", ack.disposition)
    replied = await finish_run(relay_agent, runner)
    print("answer:", replied.final_answer)
    check("hello" in replied.final_answer.lower(), "after the reply it says hello")
else:
    await runner

[meta:run_started] RunStarted(user_query='Say hello to me.', model='nvidia/nemotron-3.5-lightning-free', conversation_log=None)


2026-11-18T13:11:13.453568Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:12:00.892125Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


[thinking] We


[thinking]  need to follow


[thinking]  instruction


... (56 outputs left out)

 you today?


[meta:usage_report] UsageReport(kind='usage_report', usage={'input_tokens': 772, 'output_tokens': 169, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'thinking_tokens': 127, '_v'


[meta:run_completed] RunCompleted(stop_reason='end_turn', total_steps=2, generated_files=None, cost={'total_cost': 0.0, 'currency': 'USD', 'breakdown': {'reported_cost': 0.0}, 'run_


answer: Hello! How can I help you today?
PASS after the reply it says hello


# 9. Sub-agents

The parent's provider carries the router, so each sub-agent's model picks its own route: here the specialists run on a second model, and they exchange only the task and the answer with the parent. A sub-agent given tools on a model that failed the live tool check is refused before any request, and the refusal comes back to the parent as the tool's error.

In [11]:
from agent_base.common_tools.sub_agent_tool import SubAgentSpec

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search flights between two cities on a date."""
    return f"Flight {origin} -> {destination} on {date}: $1,200, departs 09:40."

@tool
def search_hotels(city: str, checkin: str, checkout: str) -> str:
    """Search hotels in a city between two dates."""
    return f"Hotel in {city}, {checkin} to {checkout}: $200 a night."

orchestrator = AnthropicAgent(
    system_prompt=("Delegate flights to flight_finder and hotels to hotel_finder, one call each, then give a short "
                   "plan with the total cost."),
    model=MODEL, provider_value=provider(), config=OpenAIChatLLMConfig(max_tokens=4000),
    subagents={
        "flight_finder": SubAgentSpec(name="flight_finder", description="Finds a flight.", model=SECOND_MODEL,
                                      system_prompt="Always use search_flights, then report the result.",
                                      tools=[search_flights], retry_policy=RETRY),
        "hotel_finder": SubAgentSpec(name="hotel_finder", description="Finds a hotel.", model=SECOND_MODEL,
                                     system_prompt="Always use search_hotels, then report the result.",
                                     tools=[search_hotels], retry_policy=RETRY),
    },
)
plan = await run_with_stream(orchestrator, "Plan a trip from New York to Tokyo: fly on 2026-03-15, hotel 2026-03-15 to 2026-03-20.",
                             print_items=False)
print(plan.final_answer)
check("1,200" in plan.final_answer or "1200" in plan.final_answer, "the flight's price reached the parent")
check("200" in plan.final_answer, "the hotel's price reached the parent")

refused_parent = AnthropicAgent(
    system_prompt="Delegate every sum to the adder sub-agent, then report what happened.", model=MODEL,
    provider_value=provider(), config=OpenAIChatLLMConfig(max_tokens=2000),
    subagents={"adder": SubAgentSpec(name="adder", description="Adds numbers.", system_prompt="Add.",
                                     model="zai/glm-4.7-flash", tools=[add])},
)
refused = await refused_parent.run("What is 19 + 23?")
errors = [str(b.tool_result) for m in refused_parent.agent_config.context_messages for b in m.content
          if type(b).__name__ == "ToolResultContent"]
print("\nthe sub-agent's result:", errors[:1])
check(any("failed the live tool check" in e for e in errors), "the refusal reached the parent as the tool's error")

2026-11-18T13:12:02.878213Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:12:04.669857Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:13:01.074392Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:13:03.023690Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:13:04.974342Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:13:07.321323Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


... (5 outputs left out)

2026-11-18T13:14:00.849827Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:14:02.981378Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:14:04.290943Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]



the sub-agent's result: ['[TextContent(content_block_type=<ContentBlockType.TEXT: \'text\'>, kwargs={}, text="Error: Subagent \'adder\' error: ProviderError: \'zai/glm-4.7-flash\' failed the live tool check (it answers without its tools, or its service returns calls as text), so it isn\'t given tools. Use it without tools, or replace its record with catalog.add(...) if you have seen tools work on it.")]']
PASS the refusal reached the parent as the tool's error


# 10. Compaction and externalization

Both belong to the turn loop, so they work on this provider as on Claude. **Compaction** summarises old turns once the context passes `threshold_tokens`; the summary call goes to the same model. It keeps a recent window (`preserve_recent_tokens`) and keeps whole the message where that window fills, so it needs a few turns: on the first, the one big message is that message, and nothing is older to summarise. **Externalization** moves a prompt or a tool result that is too large into a file in the agent's sandbox and gives the model a pointer to it, which it reads back with `read_file`, a few lines at a time: a whole-file read can be too large in turn and be moved to a file again. The thresholds here are tiny so both happen at once.

In [12]:
from agent_base.common_tools import ReadFileTool
from agent_base.core.hooks import HookMatcher
from agent_base.providers.anthropic import ExternalizationConfig
from agent_base.providers.anthropic.compaction import CompactionConfig

compactions = []

async def after_compact(ctx):
    compactions.append(ctx.stats if isinstance(ctx.stats, dict) else vars(ctx.stats))

note = ("Project note: the launch moved to May 12; the budget is $48,000; the lead is Priya. "
        + "Background: " + "the team reviewed vendors, timelines and risks in detail. " * 400)  # ~4,500 tokens
compacting = AnthropicAgent(
    system_prompt="Answer briefly.", model=MODEL, provider_value=provider(), config=OpenAIChatLLMConfig(max_tokens=2000),
    compaction_config=CompactionConfig(threshold_tokens=1500, preserve_recent_tokens=200),
    hooks={"after_compact": [HookMatcher(matcher=None, hooks=[after_compact])]}, sandbox_factory=sandbox_in_work,
    max_steps=6,
)
await compacting.run(note + "\n\nAcknowledge the note in one line.")
await compacting.run("In about 150 words, say why product launches often slip.")
recalled = await compacting.run("When is the launch, and what is the budget?")
texts = [getattr(b, "text", "") for m in compacting.agent_config.context_messages for b in m.content]
note_gone = not any("timelines and risks" in t for t in texts)
print("after compaction:", recalled.final_answer)
print("compactions:", compactions, "| the long note still in the context:", not note_gone)
print("the context now starts:", texts[0][:150])
check(compactions and note_gone, "the context was compacted: the long note became a summary")
check("12" in recalled.final_answer and "48" in recalled.final_answer, "the summary kept the launch date and the budget")

@tool
def large_result_tool(topic: str) -> str:
    """Look a topic up; returns a very long text."""
    lines = [f"{topic}: the key figure is 7.2 knots."] + [f"Detail line {i}: more on {topic}." for i in range(1, 1500)]
    return chr(10).join(lines)  # one line each, so read_file can read a few

externalizing = AnthropicAgent(
    system_prompt=("Use large_result_tool when asked about a topic. If its result was moved to a file, read only "
                   "its first 5 lines with read_file (limit=5), then answer briefly."), model=MODEL, max_steps=8,
    provider_value=provider(), tools=[large_result_tool, ReadFileTool().as_tool()],
    config=OpenAIChatLLMConfig(max_tokens=2000),
    externalization_config=ExternalizationConfig(max_prompt_tokens=1200, max_tool_result_tokens=1000,
                                                 max_combined_tool_result_tokens=1200),
    sandbox_factory=sandbox_in_work,
)
looked_up = await externalizing.run("Look up ocean currents with the tool and tell me the key figure.")
pointers = [i.text for m in externalizing.agent_config.context_messages for b in m.content
            if isinstance(getattr(b, "tool_result", None), list) for i in b.tool_result
            if getattr(i, "text", "").startswith("Tool result externalized to")]
print("pointer:", pointers[:1])
print("answer:", looked_up.final_answer[:160])
check(pointers, "the large tool result was moved to a file")
check("7.2" in looked_up.final_answer, "the model read the file back (the key figure, 7.2 knots)")

2026-11-18T13:14:04.718456Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:15:02.184675Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:15:02.589280Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:15:10.565716Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:15:16.155832Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:16:00.695573Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


... (2 outputs left out)

2026-11-18T13:16:05.475358Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T13:16:06.139205Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T13:17:01.028388Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


pointer: ["Tool result externalized to .context/tool_result_call-0fda47de-61f1-4910-9ae0-38ca22fb0522.txt. Use read_file with path='.context/tool_result_call-0fda47de-61f1-4910-9ae0-38ca22fb0522.txt' to access the full content."]
answer: The key figure for ocean currents is **7.2 knots**.
PASS the large tool result was moved to a file
PASS the model read the file back (the key figure, 7.2 knots)


# 11. Abort and steer

`abort()` stops a run in any phase and keeps what was complete; `steer()` replaces it with a new instruction. Shown mid-stream, mid-tool and during a frontend tool's pause.

In [13]:
verbose = "Be very detailed and verbose. Think step by step."
talker = AnthropicAgent(system_prompt=verbose, model=MODEL, provider_value=provider(), config=OpenAIChatLLMConfig(max_tokens=4000))
demo = await start_demo(talker, "Explain in great detail how to add 5 and 7, with history and examples.")
aborted = await demo.abort()
check(aborted.stop_reason == "aborted", "a streaming run aborts")

steered_agent = AnthropicAgent(system_prompt=verbose, model=MODEL, provider_value=provider(), config=OpenAIChatLLMConfig(max_tokens=4000))
demo = await start_demo(steered_agent, "Explain in great detail how to add 5 and 7, with history and examples.")
steered = await demo.steer("Stop. Give only the numeric answer.")
check(steered.stop_reason == "end_turn" and "12" in steered.final_answer, "a steer replaces the run and answers 12")

@tool
async def slow_calculate(expression: str, delay_seconds: int = 5) -> str:
    """Evaluate a math expression after a delay."""
    await asyncio.sleep(delay_seconds)
    return str(eval(expression, {"__builtins__": {}}, {}))

slow_agent = AnthropicAgent(system_prompt="Always use slow_calculate for math before answering.", model=MODEL,
                            provider_value=provider(), tools=[slow_calculate], config=OpenAIChatLLMConfig(max_tokens=2000))
demo = await start_demo(slow_agent, "Use slow_calculate to compute 25 * 4 with delay_seconds=8.", wait_for_frames=1)
await asyncio.sleep(4)
mid_tool = await demo.abort()
check(mid_tool.stop_reason == "aborted", "a run aborts mid-tool")
again = await slow_agent.run("Now compute 25 * 4 with slow_calculate, delay_seconds=1. Reply with the number.")
check("100" in again.final_answer, "the next run works after the abort")

pause_agent = AnthropicAgent(
    system_prompt="For every request, first call user_confirm with a short question. Answer only after it returns.",
    model=MODEL, provider_value=provider(), frontend_tools=[user_confirm], config=OpenAIChatLLMConfig(max_tokens=2000))
runner, pause = await run_until_pause_or_done(pause_agent, "Say hello to me.")
if pause is not None:
    paused_abort = await pause_agent.abort()
    await asyncio.gather(runner, return_exceptions=True)
    print("aborted during the pause:", paused_abort.stop_reason)
    check(paused_abort.stop_reason == "aborted" and pause_agent.agent_config.pending_relay is None,
          "an abort during a pause ends it and clears the saved pause")
else:
    await runner
    check(False, "the run paused for user_confirm (the model answered without it)")

[meta:run_started] RunStarted(user_query='Explain in great detail how to add 5 and 7, with history and examples.', model='nvidia/nemotron-3.5-lightning-free', conversation_log=Non


2026-11-18T13:17:01.472067Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


[thinking] Here


[thinking] 's a thinking


[meta:custom] Custom(name='aborted', data={'phase': 'streaming'})



stop_reason=aborted was_aborted=True abort_phase=None final_answer='Agent run was aborted by the user.'
PASS a streaming run aborts
[meta:run_started] RunStarted(user_query='Explain in great detail how to add 5 and 7, with history and examples.', model='nvidia/nemotron-3.5-lightning-free', conversation_log=Non


... (68 outputs left out)

[thinking] 


[tool_call user_confirm] {"message":"Do you want me to say hello?"}


[meta:await_input] AwaitInput(tools=[FrontendCallView(tool_use_id='call-f9bd2697-363c-459f-bcbe-540bbfc1ae31', tool_name='user_confirm', input={'message': 'Do you want me to say h


aborted during the pause: aborted
PASS an abort during a pause ends it and clears the saved pause


# 12. The bill

A gateway step costs what Vercel reported charging (`breakdown["reported_cost"]`), which is $0 on the free models; the price table is only the fallback. The settlement is what a host bills from.

In [14]:
print("basic run:      ", basic.settlement.turn_cost, "| steps:", basic.settlement.step_count)
print("orchestrator:   ", plan.settlement.turn_cost)
check("reported_cost" in basic.settlement.turn_cost.breakdown, "the bill is Vercel's reported charge")
check(basic.settlement.turn_cost.total_cost == 0, "the free model cost $0")

basic run:       CostBreakdown(total_cost=0.0, currency='USD', breakdown={'reported_cost': 0.0}, run_id=None) | steps: 3
orchestrator:    CostBreakdown(total_cost=0.0, currency='USD', breakdown={'reported_cost': 0.0}, run_id=None)
PASS the bill is Vercel's reported charge
PASS the free model cost $0


In [15]:
failed = [what for ok, what in CHECKS if not ok]
print(f"{len(CHECKS) - len(failed)} of {len(CHECKS)} checks passed")
for what in failed:
    print("  FAIL", what)
await gateway.aclose()

29 of 29 checks passed
